####Install our package

In [0]:
!pip install /Workspace/Users/gabor.szabo@allianz.com/cubix_data_engineer_capstone-0.2.9-py3-none-any.whl

####Imports

In [0]:

from cubix_data_engineer_capstone.etl.bronze.extract_and_load_file import bronze_ingest_volume

from cubix_data_engineer_capstone.etl.gold.wide_sales import get_wide_sales
from cubix_data_engineer_capstone.etl.gold.daily_product_category_metrics import get_daily_product_category_metrics
from cubix_data_engineer_capstone.etl.gold.daily_sales_metrics import get_daily_sales_metrics

from cubix_data_engineer_capstone.etl.silver.calendar import get_calendar
from cubix_data_engineer_capstone.etl.silver.customers import get_customers
from cubix_data_engineer_capstone.etl.silver.products import get_products
from cubix_data_engineer_capstone.etl.silver.product_subcategory import get_product_subcategory
from cubix_data_engineer_capstone.etl.silver.product_category import get_product_category
from cubix_data_engineer_capstone.etl.silver.sales import get_sales 
from cubix_data_engineer_capstone.etl.silver.scd import scd1_uc

from cubix_data_engineer_capstone.utils.databricks import read_file_from_volume, write_file_to_volume

In [ ]:
spark.sql("DROP CATALOG capstone CASCADE")

spark.sql("CREATE CATALOG IF NOT EXISTS capstone")
spark.sql("USE CATALOG capstone")
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")
spark.sql("CREATE VOLUME IF NOT EXISTS capstone.bronze.bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

source_system_tables = [
    "calendar",
    "customers",
    "product_category",
    "product_subcategory",
    "products",
    "sales"
]

for table in source_system_tables:
    dbutils.fs.mkdirs(f"/Volumes/capstone/bronze/bronze/{table}")

####Ingestion tasks

In [0]:
batch_2 = {
    "customers": {
        "file_name": "customers_2.csv",
        "primary_key": "CustomerKey",
        "function": get_customers
    },
    "products": {
        "file_name": "products_ingest_1.csv",
        "primary_key": "ProductKey",
        "function": get_products
    },
    "product_subcategory": {
        "file_name": "product_subcategory_ingest_1.csv",
        "primary_key": "ProductSubcategoryKey",
        "function": get_product_subcategory
    },
    "product_category": {
        "file_name": "product_category_ingest_1.csv",
        "primary_key": "ProductCategoryKey",
        "function": get_product_category
    },
    "sales": {
        "file_name": "sales_202405.csv",
        "primary_key": "SalesOrderNumber",
        "function": get_sales
    },
}

batch_3 = {
    "customers": {
        "file_name": "customers_3.csv",
        "primary_key": "CustomerKey",
        "function": get_customers
    },
    "sales": {
        "file_name": "sales_202406.csv",
        "primary_key": "SalesOrderNumber",
        "function": get_sales
    },
}

batch_4 = {
    "customers": {
        "file_name": "customers_4.csv",
        "primary_key": "CustomerKey",
        "function": get_customers

    },
    "sales": {
        "file_name": "sales_202407.csv",
        "primary_key": "SalesOrderNumber",
        "function": get_sales
    },
}

ingestion_job = batch_4

####Bronze ingestion

In [0]:
for dataset_key, params in ingestion_job.items():

    bronze_ingest_volume(
        source_path=f"/Volumes/source_system/source_system/source_files/{dataset_key}",
        bronze_path=f"/Volumes/capstone/bronze/bronze/{dataset_key}/",
        file_name=params["file_name"],
    )

    print(f"{dataset_key} ({params['file_name']}) ingested")

####Silver ingestion

In [0]:
batches = [batch_2, batch_3, batch_4]
for batch in batches:
    ingestion_job = batch
    for dataset, params in ingestion_job.items():
        raw_dataframe = read_file_from_volume(
            full_path=f"/Volumes/capstone/bronze/bronze/{dataset}/{params['file_name']}",
            format="csv"
        )
        transform_func = params["function"]
        transformed_dataframe = transform_func(raw_dataframe)
        # sales is not SCD, it's a Fact table, therefore appending.
        if dataset == "sales":
            (
                transformed_dataframe
                .write
                .format("delta")
                .mode("append")
                .saveAsTable(f"capstone.silver.{dataset}")
            )
        else:
            scd1_uc(spark, f"capstone.silver.{dataset}", transformed_dataframe, primary_key=params["primary_key"])
        print(f"{dataset} has been processed in the Silver layer.")
    # After batch_4, write wide_sales_df to parquet using write_file_to_volume function
    if batch is batch_4:
        calendar_master = spark.table("capstone.silver.calendar")
        customers_master = spark.table("capstone.silver.customers")
        product_category_master = spark.table("capstone.silver.product_category")
        product_subcategory_master = spark.table("capstone.silver.product_subcategory")
        products_master = spark.table("capstone.silver.products")
        sales_master = spark.table("capstone.silver.sales")
        wide_sales_df = get_wide_sales(
            sales_master=sales_master,
            calendar_master=calendar_master,
            customers_master=customers_master,
            products_master=products_master,
            product_subcategory_master=product_subcategory_master,
            product_category_master=product_category_master
)
        write_file_to_volume(
            df=wide_sales_df,
            full_path="results/wide_sales.parquet",
            format="parquet",
            mode="overwrite"
)

---------------------------------------------------------------------------
DateTimeException                         Traceback (most recent call last)
File <command-7963098415030722>, line 21
     13     (
     14         transformed_dataframe
     15         .write
   (...)
     18         .saveAsTable(f"capstone.silver.{dataset}")
     19     )
     20 else:
---> 21     scd1_uc(spark, f"capstone.silver.{dataset}", transformed_dataframe, primary_key=params["primary_key"])
     23 print(f"{dataset} has been processed in the Silver layer.")

File /local_disk0/.ephemeral_nfs/envs/pythonEnv-41ec29af-2433-4fc2-9f70-3938a3648a39/lib/python3.12/site-packages/cubix_data_engineer_capstone/etl/silver/scd.py:55, in scd1_uc(spark, table_name, new_data, primary_key)
     34 """
     35 Slowly Changing Dimension Type 1 for UC Volumes.
     36 Compares the master Delta table with new_data, updating or inserting as needed.
   (...)
     41 :param primary_key: Column name used as primary key.
     42

####Gold layer

In [0]:
calendar_master = spark.table("capstone.silver.calendar")
customers_master = spark.table("capstone.silver.customers")
product_category_master = spark.table("capstone.silver.product_category")
product_subcategory_master = spark.table("capstone.silver.product_subcategory")
products_master = spark.table("capstone.silver.products")
sales_master = spark.table("capstone.silver.sales")


#####Wide Sales

In [0]:
wide_sales_df = get_wide_sales(
    sales_master=sales_master,
    calendar_master=calendar_master,
    customers_master=customers_master,
    products_master=products_master,
    product_subcategory_master=product_subcategory_master,
    product_category_master=product_category_master
)

In [0]:
wide_sales_df.count()

In [0]:
display(wide_sales_df)

In [0]:
(
    wide_sales_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("capstone.gold.wide_sales")
)

In [0]:
%sql
-- Total Sales and Profit by month to identify high-performing months
SELECT 
  MonthName,
  CalendarYear,
  SUM(SalesAmount) AS TotalSales,
  SUM(Profit) AS TotalProfit
FROM
  capstone.gold.wide_sales
GROUP BY
  MonthName,
  CalendarYear,
  MonthNumberOfYear
ORDER BY
  CalendarYear,
  MonthNumberOfYear

In [0]:
%sql
-- Sales by customer demographics
SELECT 
  MaritalStatus,
  Gender,
  AVG(SalesAmount) AS AvgSales,
  COUNT(SalesOrderNumber) AS TotalOrders
FROM
  capstone.gold.wide_sales
GROUP BY
  MaritalStatus,
  Gender
ORDER BY
  AvgSales DESC

In [0]:
%sql

-- Which products are most ofer part of high-value orders.
SELECT
  ProductName,
  COUNT(SalesOrderNumber) AS HighValueOrderCount
FROM
  capstone.gold.wide_sales
WHERE
  HighValueOrder = true
GROUP BY
  ProductName
ORDER BY
  HighValueOrderCount DESC;

##### Daily Product Category Metrics

In [0]:
daily_product_category_metrics = get_daily_product_category_metrics(wide_sales_df)

In [0]:
display(daily_product_category_metrics)

In [0]:
(
    daily_product_category_metrics
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("capstone.gold.daily_product_category_metrics")
)

#####Daily Sales Metrics

In [0]:
daily_sales_metrics = get_daily_sales_metrics(wide_sales_df)

In [0]:
display(daily_sales_metrics)

In [0]:
(
    daily_sales_metrics
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("capstone.gold.daily_sales_metrics")
)

####Data Quality with The Great Expectations

In [0]:
!pip install great_expectations

In [0]:
from great_expectations.core.batch import Batch
from great_expectations.validator.validator import Validator
from great_expectations.execution_engine.sparkdf_execution_engine import SparkDFExecutionEngine
from great_expectations import get_context

In [0]:
# 1. Create a context:
context = get_context()

# 2. Convert Spark Execution Engine
execution_engine = SparkDFExecutionEngine(persist=False)

# 3. Create a Batch from the DataFrame
batch = Batch(data=wide_sales_df)

# 4. Create a Validator with the batch and execution engine
validator = Validator(execution_engine=execution_engine, batches=[batch])

# 5. Add expectations
validator.expect_column_values_to_not_be_null(
    column="SalesOrderNumber",
)

validator.expect_column_values_to_be_in_set(
    column="Gender",
    value_set=["Male", "Female"]
)

validator.expect_column_values_to_be_between(
    column="BirthDate",
    min_value='1999-01-01',
    max_value=None
)

# 6. Run validation and get results
results = validator.validate()

# 7. Process results
if results["success"]:
    print("All validations passed!")
else:
    print("Some validations failed!")
    for result in results["results"]:
        print(f"Expectation: {result['expectation_config']['type']}")
        print(f"Success: {result['success']}")
        if not result['success']:
            print(f"Details: {result['result']}")


In [0]:
validation_results = results["results"]

results_data = [
    {
        "Expectation": res["expectation_config"]["type"],
        "Column": res["expectation_config"]["kwargs"].get("column"),
        "Success": res["success"],
        "Count": res["result"].get("element_count", "N/A"),
        "Failed_Records_Count": res["result"].get("unexpected_count", "N/A"),
        "Failed_Records_Precentage": res["result"].get("unexpected_percent", "N/A")
    }
    for res in validation_results
]

# Convert results_data to Spark DataFrame for display
validation_results_df = spark.createDataFrame(results_data)

display(validation_results_df)

(
    validation_results_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("capstone.gold.wide_sales_validation_results")
)